In [1]:
import os
import pandas as pd
import numpy as np

def clean_dataframe(dataframe, daytype, direction):
    dataframe = dataframe[dataframe["DayType"] == daytype]
    dataframe = dataframe[dataframe["Direction"] == direction]
    dataframe = dataframe[dataframe["Occupation"] <= 80]
    dataframe = dataframe[dataframe["Occupation"] >= 0]
    dataframe = dataframe[dataframe["Board"] <= 80]
    dataframe = dataframe[dataframe["Board"] >= 0]
    dataframe = dataframe[dataframe["DropOff"] <= 80]
    dataframe = dataframe[dataframe["DropOff"] >= 0]
    dataframe = dataframe[dataframe["FreeDay"] == 0]

    return dataframe

DATA_DIR = "../data/E01_toA/NEW/"
IGNORES = {"RESULTS", "E01_toA.zip"}

# TripId exceeds int64 -> must stay a string or digits are silently lost.
DTYPES = {"TripId": str, "TripId.1": str, "Stop": str}


def load_all(data_dir, ignores):
    frames = []
    for name in sorted(os.listdir(data_dir)):
        if name in ignores or not name.endswith(".csv"):
            continue
        path = os.path.join(data_dir, name)          # listdir gives basenames only
        df = pd.read_csv(path, encoding="utf-8", header=0, dtype=DTYPES)
        df = clean_dataframe(df, "LA", 2)
        df["source_file"] = name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


df = load_all(DATA_DIR, IGNORES)
print(f"rows: {len(df)}   columns: {list(df.columns)}")

# ---- column names after pandas de-duplicates the repeated header fields ----
TRIP   = "TripId"
STOP   = "Stop"
STOPTS = "EventTime.1"      # stop-level timestamp (2nd EventTime)
STOPDT = "EventDate.1"
BOARD  = "Board"
DROP   = "DropOff"
OCC    = "Occupation"

# ---- order stops within each trip by the stop-level timestamp ----
df["_ts"] = pd.to_datetime(df[STOPDT] + " " + df[STOPTS], errors="coerce")
bad_ts = df["_ts"].isna().sum()
if bad_ts:
    print(f"WARNING: {bad_ts} rows with unparseable stop timestamp")

df = df.sort_values([TRIP, "_ts"]).reset_index(drop=True)
df["stop_seq"] = df.groupby(TRIP, sort=False).cumcount()

# duplicated stop visits within a trip would corrupt the recurrence
dupes = df.duplicated(subset=[TRIP, STOP]).sum()
print(f"trips: {df[TRIP].nunique()}   duplicate (trip, stop) rows: {dupes}")

g = df.groupby(TRIP, sort=False)
occ_prev = g[OCC].shift(1).fillna(0)

# A: Occupation measured on DEPARTURE (after alighting and boarding)
resid_A = df[OCC] - (occ_prev - df[DROP] + df[BOARD])

# B: Occupation measured on ARRIVAL (before alighting)
resid_B = df[OCC] - (occ_prev - g[DROP].shift(1).fillna(0)
                              + g[BOARD].shift(1).fillna(0))

for label, r in (("A departure", resid_A), ("B arrival", resid_B)):
    r = r.dropna()
    print(f"\n{label}:  exact={np.mean(r == 0):.4f}  "
          f"|r|<=1={np.mean(r.abs() <= 1):.4f}  "
          f"mean={r.mean():.3f}  std={r.std():.3f}")
    print(r.abs().describe())

# ---- trip closure: boards and dropoffs should balance over a whole trip ----
closure = g.apply(lambda t: (t[BOARD] - t[DROP]).sum())
print("\ntrip closure residual (should be ~0):")
print(closure.describe())
print(f"exactly zero: {np.mean(closure == 0):.4f}")

df.to_parquet("merged_toA.parquet", index=False)
print("\nwritten: merged_toA.parquet")

rows: 169225   columns: ['TripId', 'IdExternal', 'EventDate', 'EventTime', 'TimeSlot', 'Line', 'Direction', 'LogicVehicle', 'Weekday', 'DayType', 'Season', 'FreeDay', 'TripId.1', 'Stop', 'IdExternal.1', 'EventDate.1', 'EventTime.1', 'TimeSlot.1', 'TemperatureMax', 'TemperatureMin', 'RainProbMin', 'RainProbMax', 'Board', 'DropOff', 'Occupation', 'source_file']
trips: 12932   duplicate (trip, stop) rows: 0

A departure:  exact=0.9107  |r|<=1=0.9397  mean=0.099  std=2.403
count    169225.000000
mean          0.395663
std           2.372596
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          94.000000
dtype: float64

B arrival:  exact=0.0824  |r|<=1=0.2047  mean=-1.039  std=9.626
count    169225.000000
mean          6.508051
std           7.167660
min           0.000000
25%           2.000000
50%           4.000000
75%           9.000000
max         118.000000
dtype: float64

trip closure residual (should be ~0):
count    12932.000000
me

In [2]:
"""
Alighting model calibration for line E01.

Reads every stop CSV in a directory, cleans, reconstructs arrival occupancy,
and fits a per (Stop, TimeSlot) alighting model:

    f ~ Beta(alpha, beta)        # alighting fraction, varies between visits
    k ~ Binomial(n, f)           # n = occupancy on arrival

marginally BetaBinomial(n, alpha, beta).  Plain Binomial(n, p) is fitted as the
nested null and selected by AIC when the Beta layer does not pay for itself.

Cells with too few visits are POOLED FORWARD into the next timeslot: the merged
sample is refitted once and the resulting parameters are written to BOTH slots,
so every (Stop, TimeSlot) pair exists in the output.  The final timeslot pools
BACKWARD instead, since it has no successor.

Terminal stops are deterministic and are emitted with zeroed parameters:
    - origin terminal   -> nobody alights (bus arrives empty)
    - final terminal    -> everybody alights (handled in the simulator)
"""

import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import betabinom, binom

# --------------------------------------------------------------------------- #
# configuration
# --------------------------------------------------------------------------- #
DATA_DIR = "../data/E01_toA/NEW/"
IGNORES = {"RESULTS", "E01_toA.zip"}
OUT_CSV = "alighting_params_toA.csv"

DAYTYPE = "LA"
DIRECTION = 2
N_STOPS = 15
TIMESLOTS = list(range(5, 24))

ORIGIN_STOP = "2646"        # bus starts empty here  -> no alighting
FINAL_STOP = "5727"         # everybody off          -> deterministic

MIN_VISITS = 30             # below this a cell is pooled with a neighbour
MAX_COUNT = 80              # plausibility ceiling from clean_dataframe

DTYPES = {"TripId": str, "TripId.1": str, "Stop": str}
OCC, DROP, BOARD = "Occupation", "DropOff", "Board"


# --------------------------------------------------------------------------- #
# loading and cleaning
# --------------------------------------------------------------------------- #
def load_directory(data_dir, ignores):
    frames = []
    for name in sorted(os.listdir(data_dir)):
        if name in ignores or not name.endswith(".csv"):
            continue
        df = pd.read_csv(os.path.join(data_dir, name), encoding="utf-8",
                         header=0, dtype=DTYPES)
        df["source_file"] = name
        frames.append(df)
    if not frames:
        raise RuntimeError(f"no CSV files found in {data_dir}")
    return pd.concat(frames, ignore_index=True)


def clean(df, daytype=DAYTYPE, direction=DIRECTION):
    """Row-level filters, then DROP WHOLE TRIPS that lost any row.

    Filtering row-wise alone punches holes in otherwise valid trips, and the
    occupancy recurrence links consecutive rows -- a gap silently corrupts it.
    """
    n0 = len(df)
    keep = (
        (df["DayType"] == daytype)
        & (df["Direction"] == direction)
        & (df["FreeDay"] == 0)
        & df[OCC].between(0, MAX_COUNT)
        & df[BOARD].between(0, MAX_COUNT)
        & df[DROP].between(0, MAX_COUNT)
    )
    bad_trips = df.loc[~keep, "TripId"].unique()
    out = df[keep & ~df.TripId.isin(bad_trips)].copy()
    print(f"  cleaning: {n0} -> {len(out)} rows "
          f"({len(bad_trips)} trips discarded entirely)")
    return out


def build_visits(df, n_stops=N_STOPS):
    """Order stops within trips, keep complete trips, derive arrival occupancy."""
    df["_ts"] = pd.to_datetime(df["EventDate.1"] + " " + df["EventTime.1"],
                               errors="coerce")
    df = df.sort_values(["TripId", "_ts"]).reset_index(drop=True)

    sz = df.groupby("TripId").size()
    df = df[df.TripId.isin(sz[sz == n_stops].index)].copy()
    df["seq"] = df.groupby("TripId", sort=False).cumcount()

    # Occupation is DEPARTURE occupancy -> arrival occupancy is the previous row
    df["occ_prev"] = df.groupby("TripId", sort=False)[OCC].shift(1).fillna(0)

    print(f"  complete {n_stops}-stop trips: {df.TripId.nunique()}"
          f"  ({len(df)} rows)")

    usable = df[(df.seq.between(1, n_stops - 2))
                & (df.occ_prev > 0)
                & (df[DROP] <= df.occ_prev)].copy()
    print(f"  usable stop-visits for fitting: {len(usable)}")
    return df, usable


# --------------------------------------------------------------------------- #
# distribution fitting
# --------------------------------------------------------------------------- #
def fit_binomial(n, k):
    p = float(np.clip(k.sum() / n.sum(), 1e-9, 1 - 1e-9))
    ll = float(binom.logpmf(k, n, p).sum())
    return {"p": p, "loglik": ll, "aic": 2 - 2 * ll}


def _moment_seed(n, k):
    f = k / n
    mu = float(np.clip(f.mean(), 1e-3, 1 - 1e-3))
    denom = mu * (1 - mu) / n.mean()
    vr = f.var(ddof=1) / denom if denom > 0 else 1.0
    rho = float(np.clip((vr - 1) / max(n.mean() - 1, 1), 1e-4, 0.5))
    s = 1 / rho - 1
    return mu * s, (1 - mu) * s


def fit_betabinom(n, k):
    def nll(theta):
        a, b = np.exp(theta)
        if not np.isfinite(a + b) or a + b > 1e6:
            return 1e12
        return -float(betabinom.logpmf(k, n, a, b).sum())

    best = None
    for seed in (_moment_seed(n, k), (1.0, 1.0), (5.0, 20.0)):
        try:
            r = minimize(nll, np.log(np.clip(seed, 1e-3, 1e5)),
                         method="Nelder-Mead",
                         options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-8})
            if best is None or r.fun < best.fun:
                best = r
        except Exception:
            continue
    a, b = (float(v) for v in np.exp(best.x))
    ll = -float(best.fun)
    return {"alpha": a, "beta": b, "loglik": ll, "aic": 4 - 2 * ll,
            "mu": a / (a + b), "conc": a + b, "rho": 1 / (a + b + 1)}


def fit_cell(n, k, stop, slot, visits, pooled_from=""):
    """Fit both families on one sample and select by AIC."""
    bi, bb = fit_binomial(n, k), fit_betabinom(n, k)
    delta = bi["aic"] - bb["aic"]                 # > 0 favours beta-binomial
    use_bb = delta > 2
    return {
        "Stop": stop, "TimeSlot": slot, "visits": visits,
        "n_mean": float(n.mean()),
        "family": "betabinom" if use_bb else "binomial",
        # parameters the simulator reads: for the binomial case the Beta is
        # degenerate at p, written as a large concentration so a single sampling
        # path (draw f from Beta, then k from Binomial) works for both families.
        "alpha": bb["alpha"] if use_bb else bi["p"] * 1e6,
        "beta": bb["beta"] if use_bb else (1 - bi["p"]) * 1e6,
        "mu": bb["mu"] if use_bb else bi["p"],
        "conc": bb["conc"] if use_bb else 1e6,
        "rho": bb["rho"] if use_bb else 0.0,
        "p_binom": bi["p"],
        "aic_bb": bb["aic"], "aic_bin": bi["aic"], "delta_aic": delta,
        "pooled_from": pooled_from,
        "status": "pooled" if pooled_from else "ok",
    }


# --------------------------------------------------------------------------- #
# per-stop fitting with forward pooling
# --------------------------------------------------------------------------- #
def fit_stop(sub, stop, timeslots=TIMESLOTS):
    """Fit every timeslot for one stop, pooling thin cells forward.

    A cell with fewer than MIN_VISITS visits is merged with the NEXT timeslot,
    refitted on the combined sample, and the same parameters are written to both
    slots.  The last timeslot has no successor, so it pools BACKWARD instead.
    """
    by_slot = {s: g for s, g in sub.groupby("TimeSlot")}
    rows, pending, pending_slots = [], None, []

    for slot in timeslots:
        g = by_slot.get(slot)
        cur = g[["occ_prev", DROP]].to_numpy(float) if g is not None else np.empty((0, 2))
        acc = cur if pending is None else np.vstack([pending, cur])
        slots = pending_slots + [slot]

        if len(acc) >= MIN_VISITS:
            n, k = acc[:, 0], acc[:, 1]
            tag = "+".join(str(s) for s in slots) if len(slots) > 1 else ""
            fit = fit_cell(n, k, stop, slot, len(acc), tag)
            for s in slots:                        # same parameters to every slot
                r = dict(fit); r["TimeSlot"] = s
                rows.append(r)
            pending, pending_slots = None, []
        else:
            pending, pending_slots = acc, slots     # carry forward

    # leftover tail: pool BACKWARD into the last successfully fitted cell
    if pending_slots:
        done = pd.DataFrame(rows)
        if len(done):
            last = done.sort_values("TimeSlot").iloc[-1]
            for s in pending_slots:
                r = last.to_dict()
                r.update({"TimeSlot": s, "status": "pooled",
                          "pooled_from": f"back<-{int(last.TimeSlot)}"})
                rows.append(r)
        else:                                       # stop never had enough data
            for s in pending_slots:
                rows.append({"Stop": stop, "TimeSlot": s, "visits": 0,
                             "family": "binomial", "alpha": 0.0, "beta": 1e6,
                             "mu": 0.0, "conc": 1e6, "rho": 0.0, "p_binom": 0.0,
                             "status": "empty", "pooled_from": ""})
    return rows


def terminal_rows(stop, kind, timeslots=TIMESLOTS):
    """Deterministic stops, emitted for coherence so no cell is ever missing."""
    return [{"Stop": stop, "TimeSlot": s, "visits": 0, "n_mean": 0.0,
             "family": "deterministic", "alpha": 0.0, "beta": 0.0,
             "mu": 0.0 if kind == "origin" else 1.0, "conc": 0.0, "rho": 0.0,
             "p_binom": 0.0 if kind == "origin" else 1.0,
             "aic_bb": np.nan, "aic_bin": np.nan, "delta_aic": np.nan,
             "pooled_from": "", "status": f"terminal_{kind}"}
            for s in timeslots]


# --------------------------------------------------------------------------- #
def main():
    print(f"reading {DATA_DIR}")
    df = load_directory(DATA_DIR, IGNORES)
    print(f"  raw rows: {len(df)}")
    df = clean(df)
    full, usable = build_visits(df)

    rows = []
    rows += terminal_rows(ORIGIN_STOP, "origin")
    rows += terminal_rows(FINAL_STOP, "final")

    for stop, sub in usable.groupby("Stop"):
        if stop in (ORIGIN_STOP, FINAL_STOP):
            continue
        rows += fit_stop(sub, stop)

    res = pd.DataFrame(rows).sort_values(["Stop", "TimeSlot"]).reset_index(drop=True)

    cols = ["Stop", "TimeSlot", "family", "alpha", "beta", "mu", "conc", "rho",
            "p_binom", "visits", "n_mean", "aic_bb", "aic_bin", "delta_aic",
            "pooled_from", "status"]
    res = res[cols]
    res.to_csv(OUT_CSV, index=False)

    fitted = res[res.family.isin(["betabinom", "binomial"])]
    print(f"\ncells written: {len(res)}  "
          f"(expected {res.Stop.nunique()} stops x {len(TIMESLOTS)} slots "
          f"= {res.Stop.nunique() * len(TIMESLOTS)})")
    print(f"  beta-binomial : {(fitted.family == 'betabinom').sum()}")
    print(f"  binomial      : {(fitted.family == 'binomial').sum()}")
    print(f"  deterministic : {(res.family == 'deterministic').sum()}")
    print(f"  pooled cells  : {(res.status == 'pooled').sum()}")
    print(f"\nwritten: {OUT_CSV}")


if __name__ == "__main__":
    main()

reading ../data/E01_toA/NEW/
  raw rows: 269346
  cleaning: 269346 -> 164845 rows (23185 trips discarded entirely)
  complete 15-stop trips: 6629  (99435 rows)
  usable stop-visits for fitting: 85560

cells written: 285  (expected 15 stops x 19 slots = 285)
  beta-binomial : 227
  binomial      : 20
  deterministic : 38
  pooled cells  : 39

written: alighting_params_toA.csv


In [ ]:
"""
Alighting model calibration for line E01.

Reads every stop CSV in a directory, cleans, reconstructs arrival occupancy,
and fits a per (Stop, TimeSlot) alighting model:

    f ~ Beta(alpha, beta)        # alighting fraction, varies between visits
    k ~ Binomial(n, f)           # n = occupancy on arrival

marginally BetaBinomial(n, alpha, beta).  Plain Binomial(n, p) is fitted as the
nested null and selected by AIC when the Beta layer does not pay for itself.

Cells with too few visits are POOLED FORWARD into the next timeslot: the merged
sample is refitted once and the resulting parameters are written to BOTH slots,
so every (Stop, TimeSlot) pair exists in the output.  The final timeslot pools
BACKWARD instead, since it has no successor.

Terminal stops are deterministic and are emitted with zeroed parameters:
    - origin terminal   -> nobody alights (bus arrives empty)
    - final terminal    -> everybody alights (handled in the simulator)
"""

import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import betabinom, binom

# --------------------------------------------------------------------------- #
# configuration
# --------------------------------------------------------------------------- #
DATA_DIR = "../data/E01_toB/"
IGNORES = {"RESULTS", "E01_toB.zip"}
OUT_CSV = "alighting_params_toB.csv"

DAYTYPE = "LA"
DIRECTION = 1
N_STOPS = 12
TIMESLOTS = list(range(5, 24))

ORIGIN_STOP = "5727"        # bus starts empty here  -> no alighting
FINAL_STOP = "2646"         # everybody off          -> deterministic

MIN_VISITS = 30             # below this a cell is pooled with a neighbour
MAX_COUNT = 80              # plausibility ceiling from clean_dataframe

DTYPES = {"TripId": str, "TripId.1": str, "Stop": str}
OCC, DROP, BOARD = "Occupation", "DropOff", "Board"


# --------------------------------------------------------------------------- #
# loading and cleaning
# --------------------------------------------------------------------------- #
def load_directory(data_dir, ignores):
    frames = []
    for name in sorted(os.listdir(data_dir)):
        if name in ignores or not name.endswith(".csv"):
            continue
        df = pd.read_csv(os.path.join(data_dir, name), encoding="utf-8",
                         header=0, dtype=DTYPES)
        df["source_file"] = name
        frames.append(df)
    if not frames:
        raise RuntimeError(f"no CSV files found in {data_dir}")
    return pd.concat(frames, ignore_index=True)


def clean(df, daytype=DAYTYPE, direction=DIRECTION):
    """Row-level filters, then DROP WHOLE TRIPS that lost any row.

    Filtering row-wise alone punches holes in otherwise valid trips, and the
    occupancy recurrence links consecutive rows -- a gap silently corrupts it.
    """
    n0 = len(df)
    keep = (
        (df["DayType"] == daytype)
        & (df["Direction"] == direction)
        & (df["FreeDay"] == 0)
        & df[OCC].between(0, MAX_COUNT)
        & df[BOARD].between(0, MAX_COUNT)
        & df[DROP].between(0, MAX_COUNT)
    )
    bad_trips = df.loc[~keep, "TripId"].unique()
    out = df[keep & ~df.TripId.isin(bad_trips)].copy()
    print(f"  cleaning: {n0} -> {len(out)} rows "
          f"({len(bad_trips)} trips discarded entirely)")
    return out


def build_visits(df, n_stops=N_STOPS):
    """Order stops within trips, keep complete trips, derive arrival occupancy."""
    df["_ts"] = pd.to_datetime(df["EventDate.1"] + " " + df["EventTime.1"],
                               errors="coerce")
    df = df.sort_values(["TripId", "_ts"]).reset_index(drop=True)

    sz = df.groupby("TripId").size()
    df = df[df.TripId.isin(sz[sz == n_stops].index)].copy()
    df["seq"] = df.groupby("TripId", sort=False).cumcount()

    # Occupation is DEPARTURE occupancy -> arrival occupancy is the previous row
    df["occ_prev"] = df.groupby("TripId", sort=False)[OCC].shift(1).fillna(0)

    print(f"  complete {n_stops}-stop trips: {df.TripId.nunique()}"
          f"  ({len(df)} rows)")

    usable = df[(df.seq.between(1, n_stops - 2))
                & (df.occ_prev > 0)
                & (df[DROP] <= df.occ_prev)].copy()
    print(f"  usable stop-visits for fitting: {len(usable)}")
    return df, usable


# --------------------------------------------------------------------------- #
# distribution fitting
# --------------------------------------------------------------------------- #
def fit_binomial(n, k):
    p = float(np.clip(k.sum() / n.sum(), 1e-9, 1 - 1e-9))
    ll = float(binom.logpmf(k, n, p).sum())
    return {"p": p, "loglik": ll, "aic": 2 - 2 * ll}


def _moment_seed(n, k):
    f = k / n
    mu = float(np.clip(f.mean(), 1e-3, 1 - 1e-3))
    denom = mu * (1 - mu) / n.mean()
    vr = f.var(ddof=1) / denom if denom > 0 else 1.0
    rho = float(np.clip((vr - 1) / max(n.mean() - 1, 1), 1e-4, 0.5))
    s = 1 / rho - 1
    return mu * s, (1 - mu) * s


def fit_betabinom(n, k):
    def nll(theta):
        a, b = np.exp(theta)
        if not np.isfinite(a + b) or a + b > 1e6:
            return 1e12
        return -float(betabinom.logpmf(k, n, a, b).sum())

    best = None
    for seed in (_moment_seed(n, k), (1.0, 1.0), (5.0, 20.0)):
        try:
            r = minimize(nll, np.log(np.clip(seed, 1e-3, 1e5)),
                         method="Nelder-Mead",
                         options={"maxiter": 4000, "xatol": 1e-8, "fatol": 1e-8})
            if best is None or r.fun < best.fun:
                best = r
        except Exception:
            continue
    a, b = (float(v) for v in np.exp(best.x))
    ll = -float(best.fun)
    return {"alpha": a, "beta": b, "loglik": ll, "aic": 4 - 2 * ll,
            "mu": a / (a + b), "conc": a + b, "rho": 1 / (a + b + 1)}


def fit_cell(n, k, stop, slot, visits, pooled_from=""):
    """Fit both families on one sample and select by AIC."""
    bi, bb = fit_binomial(n, k), fit_betabinom(n, k)
    delta = bi["aic"] - bb["aic"]                 # > 0 favours beta-binomial
    use_bb = delta > 2
    return {
        "Stop": stop, "TimeSlot": slot, "visits": visits,
        "n_mean": float(n.mean()),
        "family": "betabinom" if use_bb else "binomial",
        # parameters the simulator reads: for the binomial case the Beta is
        # degenerate at p, written as a large concentration so a single sampling
        # path (draw f from Beta, then k from Binomial) works for both families.
        "alpha": bb["alpha"] if use_bb else bi["p"] * 1e6,
        "beta": bb["beta"] if use_bb else (1 - bi["p"]) * 1e6,
        "mu": bb["mu"] if use_bb else bi["p"],
        "conc": bb["conc"] if use_bb else 1e6,
        "rho": bb["rho"] if use_bb else 0.0,
        "p_binom": bi["p"],
        "aic_bb": bb["aic"], "aic_bin": bi["aic"], "delta_aic": delta,
        "pooled_from": pooled_from,
        "status": "pooled" if pooled_from else "ok",
    }


# --------------------------------------------------------------------------- #
# per-stop fitting with forward pooling
# --------------------------------------------------------------------------- #
def fit_stop(sub, stop, timeslots=TIMESLOTS):
    """Fit every timeslot for one stop, pooling thin cells forward.

    A cell with fewer than MIN_VISITS visits is merged with the NEXT timeslot,
    refitted on the combined sample, and the same parameters are written to both
    slots.  The last timeslot has no successor, so it pools BACKWARD instead.
    """
    by_slot = {s: g for s, g in sub.groupby("TimeSlot")}
    rows, pending, pending_slots = [], None, []

    for slot in timeslots:
        g = by_slot.get(slot)
        cur = g[["occ_prev", DROP]].to_numpy(float) if g is not None else np.empty((0, 2))
        acc = cur if pending is None else np.vstack([pending, cur])
        slots = pending_slots + [slot]

        if len(acc) >= MIN_VISITS:
            n, k = acc[:, 0], acc[:, 1]
            tag = "+".join(str(s) for s in slots) if len(slots) > 1 else ""
            fit = fit_cell(n, k, stop, slot, len(acc), tag)
            for s in slots:                        # same parameters to every slot
                r = dict(fit); r["TimeSlot"] = s
                rows.append(r)
            pending, pending_slots = None, []
        else:
            pending, pending_slots = acc, slots     # carry forward

    # leftover tail: pool BACKWARD into the last successfully fitted cell
    if pending_slots:
        done = pd.DataFrame(rows)
        if len(done):
            last = done.sort_values("TimeSlot").iloc[-1]
            for s in pending_slots:
                r = last.to_dict()
                r.update({"TimeSlot": s, "status": "pooled",
                          "pooled_from": f"back<-{int(last.TimeSlot)}"})
                rows.append(r)
        else:                                       # stop never had enough data
            for s in pending_slots:
                rows.append({"Stop": stop, "TimeSlot": s, "visits": 0,
                             "family": "binomial", "alpha": 0.0, "beta": 1e6,
                             "mu": 0.0, "conc": 1e6, "rho": 0.0, "p_binom": 0.0,
                             "status": "empty", "pooled_from": ""})
    return rows


def terminal_rows(stop, kind, timeslots=TIMESLOTS):
    """Deterministic stops, emitted for coherence so no cell is ever missing."""
    return [{"Stop": stop, "TimeSlot": s, "visits": 0, "n_mean": 0.0,
             "family": "deterministic", "alpha": 0.0, "beta": 0.0,
             "mu": 0.0 if kind == "origin" else 1.0, "conc": 0.0, "rho": 0.0,
             "p_binom": 0.0 if kind == "origin" else 1.0,
             "aic_bb": np.nan, "aic_bin": np.nan, "delta_aic": np.nan,
             "pooled_from": "", "status": f"terminal_{kind}"}
            for s in timeslots]


# --------------------------------------------------------------------------- #
def main():
    print(f"reading {DATA_DIR}")
    df = load_directory(DATA_DIR, IGNORES)
    print(f"  raw rows: {len(df)}")
    df = clean(df)
    full, usable = build_visits(df)

    rows = []
    rows += terminal_rows(ORIGIN_STOP, "origin")
    rows += terminal_rows(FINAL_STOP, "final")

    for stop, sub in usable.groupby("Stop"):
        if stop in (ORIGIN_STOP, FINAL_STOP):
            continue
        rows += fit_stop(sub, stop)

    res = pd.DataFrame(rows).sort_values(["Stop", "TimeSlot"]).reset_index(drop=True)

    cols = ["Stop", "TimeSlot", "family", "alpha", "beta", "mu", "conc", "rho",
            "p_binom", "visits", "n_mean", "aic_bb", "aic_bin", "delta_aic",
            "pooled_from", "status"]
    res = res[cols]
    res.to_csv(OUT_CSV, index=False)

    fitted = res[res.family.isin(["betabinom", "binomial"])]
    print(f"\ncells written: {len(res)}  "
          f"(expected {res.Stop.nunique()} stops x {len(TIMESLOTS)} slots "
          f"= {res.Stop.nunique() * len(TIMESLOTS)})")
    print(f"  beta-binomial : {(fitted.family == 'betabinom').sum()}")
    print(f"  binomial      : {(fitted.family == 'binomial').sum()}")
    print(f"  deterministic : {(res.family == 'deterministic').sum()}")
    print(f"  pooled cells  : {(res.status == 'pooled').sum()}")
    print(f"\nwritten: {OUT_CSV}")

if __name__ == "__main__":
    main()

reading ../data/E01_toB/
  raw rows: 217189
  cleaning: 217189 -> 127034 rows (23277 trips discarded entirely)
  complete 12-stop trips: 6267  (75204 rows)
  usable stop-visits for fitting: 62301

cells written: 228  (expected 12 stops x 19 slots = 228)
  beta-binomial : 176
  binomial      : 14
  deterministic : 38
  pooled cells  : 10

written: alighting_params_toB.csv


In [ ]:
#Now, below all this analysis, save whats important instead of all of that bullshit



